In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm.auto import tqdm
import sacrebleu
from comet import download_model, load_from_checkpoint
import os

from model_decoder_only import TransformerDecoderOnly

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_PATH = "checkpoint_decoder_only.pth" 
TEST_DATA_PATH = "../data/test_data.tsv"
TOKENIZER_PATH = "tokenizer_decoder_only.json"
RESULTS_FILE = "results_decoder_only.txt"

D_MODEL = 256          
N_LAYERS = 4           
N_HEAD = 8
MAX_LEN = 2000
DROPOUT = 0.1

class TestDataset(Dataset):
    def __init__(self, path, tokenizer_path):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.pairs = []
        
        self.sos_id = self.tokenizer.token_to_id("<sos>")
        self.sep_id = self.tokenizer.token_to_id("<sep>")
        if self.sep_id is None:
             self.sep_id = self.tokenizer.token_to_id("<eos>")
        
        print(f"Reading test data from {path}...")
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 4:
                    self.pairs.append((parts[1], parts[3]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_ids = self.tokenizer.encode(src_text).ids
        prompt_tensor = torch.tensor([self.sos_id] + src_ids + [self.sep_id], dtype=torch.long)
        
        return prompt_tensor, src_text, tgt_text

def collate_test(batch):
    src_tensors, src_texts, tgt_texts = zip(*batch)
    pad_id = 0 
    src_padded = torch.nn.utils.rnn.pad_sequence(src_tensors, padding_value=pad_id, batch_first=True)
    return src_padded, src_texts, tgt_texts

def generate_translations(model, loader, tokenizer, device, max_new_tokens=100):
    model.eval()
    generated_texts = []
    reference_texts = []
    source_texts = []
    
    eos_id = tokenizer.token_to_id("<eos>")
    sep_id = tokenizer.token_to_id("<sep>")
    if sep_id is None: sep_id = tokenizer.token_to_id("<eos>")
    
    print("Generating translations...")
    
    with torch.no_grad():
        for prompt_tensor, src_txt, tgt_txt in tqdm(loader):
            curr_seq = prompt_tensor.to(device)
            batch_size = curr_seq.size(0)
            
            sep_positions = []
            for i in range(batch_size):
                pad_id = 0
                seq = prompt_tensor[i].tolist()

                try:
                    first_pad_idx = seq.index(pad_id)
                    actual_len = first_pad_idx
                except ValueError:
                    actual_len = len(seq)
                
                sep_positions.append(actual_len - 1)
            
            finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
            
            for _ in range(max_new_tokens):
                logits = model(curr_seq)
                
                next_token_logits = logits[:, -1, :]
                next_token = next_token_logits.argmax(dim=-1).unsqueeze(1) # [B, 1]
                
                curr_seq = torch.cat([curr_seq, next_token], dim=1)
                
                is_eos = (next_token.squeeze(1) == eos_id)
                finished = finished | is_eos
                
                if finished.all():
                    break
            
            decoded_batch = []
            for i in range(batch_size):
                full_seq = curr_seq[i].tolist()
                
                try:
                    full_text = tokenizer.decode(full_seq, skip_special_tokens=False)
                    
                    if full_text.startswith("<sos>"):
                        full_text = full_text.replace("<sos>", "", 1)
                        
                    if "<sep>" in full_text:
                        parts = full_text.split("<sep>")
                        if len(parts) > 1:
                            translation = parts[1]
                        else:
                            translation = ""
                    elif "<eos>" in full_text: 
                         parts = full_text.split("<eos>")
                         if len(parts) > 1:
                            translation = parts[1]
                         else:
                            translation = parts[0]
                    else:
                        translation = full_text

                    translation = translation.replace("<eos>", "").replace("<pad>", "").strip()
                    
                except Exception as e:
                    print(f"Error decoding: {e}")
                    translation = ""
                
                decoded_batch.append(translation)
            
            generated_texts.extend(decoded_batch)
            reference_texts.extend(tgt_txt)
            source_texts.extend(src_txt)
            
    return source_texts, generated_texts, reference_texts

if __name__ == "__main__":
    if not os.path.exists(TOKENIZER_PATH):
        raise FileNotFoundError("Tokenizer not found!")
        
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
    vocab_size = tokenizer.get_vocab_size()
    pad_idx = tokenizer.token_to_id("<pad>")
    
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"Warning: {CHECKPOINT_PATH} not found. Metrics will be random.")

    print("Loading Decoder-Only model...")
    model = TransformerDecoderOnly(
        vocab_size=vocab_size,
        d_model=D_MODEL,
        n_layer=N_LAYERS,
        n_head=N_HEAD,
        d_head=D_MODEL // N_HEAD,
        d_ff=D_MODEL * 4,
        max_len=MAX_LEN,
        dropout=DROPOUT,
        pad_idx=pad_idx
    ).to(DEVICE)
    
    if os.path.exists(CHECKPOINT_PATH):
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Model loaded from epoch {checkpoint['epoch']}")
    
    test_dataset = TestDataset(TEST_DATA_PATH, TOKENIZER_PATH)
    test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_test)

    sources, hypotheses, references = generate_translations(model, test_loader, tokenizer, DEVICE)

    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        for src, hyp, ref in zip(sources, hypotheses, references):
            f.write(f"SRC: {src}\nREF: {ref}\nHYP: {hyp}\n{'-'*20}\n")
    print(f"Translations saved to {RESULTS_FILE}")

    print("\nCalculating metrics...")
    
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    print(f"BLEU: {bleu.score:.2f}")
    
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    print(f"ChrF++: {chrf.score:.2f}")

    try:
        print("Calculating COMET...")
        model_path = download_model("Unbabel/wmt22-comet-da")
        comet_model = load_from_checkpoint(model_path)
        
        data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(sources, hypotheses, references)]
        comet_score = comet_model.predict(data, batch_size=16, gpus=1 if torch.cuda.is_available() else 0)
        print(f"COMET: {comet_score.system_score:.4f}")
    except Exception as e:
        print(f"COMET calculation failed: {e}")


/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/lib/python3.10/dist-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-26 12:33:40.376534: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-26 12:3

Loading Decoder-Only model...
Model loaded from epoch 19
Reading test data from data/test_data.tsv...
Generating translations...


100%|██████████| 938/938 [01:37<00:00,  9.62it/s]


Translations saved to results_decoder_only.txt

Calculating metrics...


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


BLEU: 24.32
ChrF++: 43.56
Calculating COMET...


Fetching 5 files: 100%|██████████| 5/5 [00:11<00:00,  2.26s/it]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../tmp/xdg_cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/jupyter/.local/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize

COMET: 0.6820


In [2]:
import json
from datetime import datetime

# Собираем словарь с результатами
metrics_log = {
    "model_type": "Decoder-Only Transformer (GPT-style)",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_file": TEST_DATA_PATH,
    "metrics": {
        "BLEU": round(bleu.score, 2),
        "ChrF++": round(chrf.score, 2),
        "COMET": round(comet_score.system_score, 4) if 'comet_score' in locals() else None
    },
    "config": {
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_head": N_HEAD
    }
}

filename = "fancy_metrics_decoder_only.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(metrics_log, f, indent=4, ensure_ascii=False)

print(f"✅ Метрики успешно сохранены в файл: {filename}")
print("-" * 30)
print(json.dumps(metrics_log, indent=4, ensure_ascii=False))


✅ Метрики успешно сохранены в файл: fancy_metrics_decoder_only.json
------------------------------
{
    "model_type": "Decoder-Only Transformer (GPT-style)",
    "timestamp": "2025-11-26 12:38:33",
    "dataset_file": "data/test_data.tsv",
    "metrics": {
        "BLEU": 24.32,
        "ChrF++": 43.56,
        "COMET": 0.682
    },
    "config": {
        "d_model": 256,
        "n_layers": 4,
        "n_head": 8
    }
}
